# iLILA / LDC-IA proposal-aligned technical noise budget

This notebook follows the current August 2026 Templeton proposal rather than the earlier 5 km single-arm sensitivity concept.

## Controlling configuration

- **iLILA:** 1 km equal-arm, dark-fringe, LIGO-like Lunar Deformation Channel (LDC)
- **Laboratory LDC-IA:** 1 m arms with emulated far reflectors
- **Verification band:** 1–100 mHz
- **Preliminary displacement requirement:** $10^{-15}$ m/√Hz
- **Optics:** 1064 nm, 1 W laser amplified to 2 W, reference-cavity stabilization, mode cleaners, MoonLIGHT 100 mm corner-cube reflectors
- **Thermal requirement:** optical bench maintained within ±10 mK (a stability bound, not an ASD)
- **Seismic reference:** LOVBB target $10^{-11}$ m s$^{-2}$/√Hz, described in the proposal as approximately $10^{-15}$ m/√Hz strainmeter-equivalent displacement in band

The proposal assigns the actual Technical Noise Budget and Closure Plan to WP I.C. Therefore the notebook does not disguise missing inputs as settled requirements.

## Visual encoding

- Dark solid: current proposal requirement or directly specified performance
- Pale solid: older LILA sensitivity-paper heritage retained for traceability
- Dashed: explicit, editable assumption
- Dotted: diagnostic upper limit, not measured lunar background


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from lila_pioneer_noise_budget_V2 import (
    CURVE_STATUS,
    LIMITATIONS,
    LILAParameters,
    assumptions_table,
    calculate_noise_curves,
    configuration_summary,
    curves_dataframe,
    frequency_grid,
    plot_noise_budget,
    reference_table,
)


## 1. Proposal configurations


In [ ]:
p = LILAParameters()
configuration_summary(p)


## 2. Parameters and assumptions

The parameter table identifies which values come directly from the current proposal, which are inherited from the older LILA sensitivity paper, and which are editable assumptions. Key unknowns include the 1 km optical link budget, received photodiode power, arm-length mismatch, frequency-stabilization spectrum, thermal ASD, alignment noise, photoreceiver/digitizer spectrum, and disturbance-subtraction transfer function.


In [ ]:
# Example sensitivity studies:
# p.assumed_round_trip_optical_efficiency = 0.01
# p.assumed_arm_length_mismatch_m = 0.10
# p.assumed_temperature_asd_0p1Hz_K_per_rtHz = 1e-8
# p.assumed_seismic_subtraction_residual_fraction = 0.01

assumptions_table(p)


## 3. Calculate and plot


In [ ]:
f = frequency_grid(p)
curves = calculate_noise_curves(f, p)
fig = plot_noise_budget(f, curves, p)
plt.show()


## 4. Reference values

Values are equivalent differential-displacement ASDs in m/√Hz. `heritage_thermal_conservative` is an alternative scenario and is not double-counted. Seismic-reference curves are not included in the provisional instrument total.


In [ ]:
table = reference_table(f, curves)
table.style.format("{:.3e}", na_rep="—")


## 5. Curve provenance and confidence


In [ ]:
pd.DataFrame(
    [{"curve": name, "status": status} for name, status in CURVE_STATUS.items()]
)


## 6. Known shortcomings and closure path


In [ ]:
pd.DataFrame(
    {"priority": range(1, len(LIMITATIONS) + 1), "limitation_or_needed_improvement": LIMITATIONS}
)


## 7. Export the current realization


In [ ]:
output_dir = Path("lila_noise_outputs")
output_dir.mkdir(exist_ok=True)

plot_noise_budget(f, curves, p, output_dir / "lila_pioneer_noise_budget_V2.png")
plt.close("all")
curves_dataframe(f, curves).to_csv(output_dir / "lila_pioneer_noise_budget_data_V2.csv", index=False)
reference_table(f, curves).to_csv(output_dir / "lila_pioneer_reference_values_V2.csv")
assumptions_table(p).to_csv(output_dir / "lila_pioneer_model_assumptions_V2.csv", index=False)

print(f"Saved outputs to {output_dir.resolve()}")


## Source hierarchy

1. **Controlling design:** *Laser Interferometer Lunar Antenna (LILA): Listening for the First Cosmic Sound*, Templeton proposal, August 2026.
2. **Heritage sensitivity estimates:** T. Creighton et al., *Fundamental Noise and Gravitational-Wave Sensitivity of the Laser Interferometer Lunar Antenna (LILA)*, arXiv:2508.18437.
3. **Mission background:** K. Jani et al., *Laser Interferometer Lunar Antenna (LILA): Advancing the U.S. Priorities in Gravitational-wave and Lunar Science*, arXiv:2508.11631.
4. **Laser-noise method:** J. B. Camp et al., *Analysis of light noise sources in a recycled Michelson interferometer with Fabry–Perot arms*, JOSA A 17, 120–128 (2000).
5. G. Harry, *Coating thermal noise formulas*, LIGO-T0900161.
